# 01 — Raw Ingestion, Time Handling, & Exploratory Data Analysis

This notebook performs the initial preprocessing of the raw dataset.

Main objectives:
1. Download long-term field monitoring data from the server
2. Exploratory data analysis
3. Convert raw timestamps to datetime format
4. Create local timezone-aware timestamps
5. Define irradiance measurement correctly (POA global)
6. Generate consistent time features
7. Align all data to a common 15-minute time bin

In [1]:
# Install parquet support if needed
!pip install -q pyarrow

In [2]:
import pyodbc 
import os
import glob
from pathlib import Path
import getpass
import pandas as pd
import numpy as np
import json
from IPython.display import display

pyodbc.drivers() #ODBC Driver 18 for SQL Server needed for data extraction

['SQL Server',
 'ODBC Driver 18 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']

## 1. Downloading long-term field-montioring data from the DTU Server 

In [ ]:
# Database connection settings
# Note: DTU VPN must be active before running this cell (use DTU Learn credentials and provide Microsoft authentication code).

server = getpass.getpass("Enter server name:")
database = getpass.getpass("Enter database name:")
username = getpass.getpass("Enter database username:")
password = getpass.getpass("Enter database password: ")

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)
print("Connected to database")

In [3]:
tables = pd.read_sql("""
SELECT TABLE_SCHEMA, TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_SCHEMA, TABLE_NAME;
""", conn)

tables.head(20)

C:\Users\amina\AppData\Local\Temp\ipykernel_43340\1392725072.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql("""


,TABLE_SCHEMA,TABLE_NAME
0,dbo,IV_Curves_RAW
1,dbo,IV_Curves_RAW_new
2,dbo,IV_Curves_RAW_temp
3,dbo,IV_Summary
4,dbo,LNDBColumnMeta
5,dbo,LNDBStationMeta
6,dbo,LNDBTableMeta
7,dbo,Z_Curves_RAW
8,dbo,Z_Curves_RAW_new
9,dbo,Z_Summary


In [ ]:
# Preview the IV_Summary table structure
df_preview = pd.read_sql(
    """
    SELECT TOP (1000) *
    FROM dbo.IV_Summary
    ORDER BY UnixTime, Module_ID;
    """,
    conn
)

df_preview.info()
display(df_preview.head())

In [ ]:
# Retrieve table-level metadata before full extraction
#1. total number of rows in the SQL table 
#2. first and last timestamp in the table 
#3. lowest and highest Module ID in the table 
#4. number of unique modules in the table
meta = pd.read_sql(
    """
    SELECT 
        COUNT(*) AS n_rows,
        MIN(UnixTime) AS min_unix,
        MAX(UnixTime) AS max_unix,
        MIN(Module_ID) AS min_module,
        MAX(Module_ID) AS max_module,
        COUNT(DISTINCT Module_ID) AS n_modules
    FROM dbo.IV_Summary;
    """,
    conn
)

display(meta)

min_unix = int(meta.loc[0, "min_unix"]) #first time entry in Unix Time 
max_unix = int(meta.loc[0, "max_unix"]) #last time entry in Unix Time

#Database time range (EDA)
print(
    "Database time range:",
    pd.to_datetime(min_unix, unit="s", utc=True),
    "to",
    pd.to_datetime(max_unix, unit="s", utc=True)
)

In [ ]:
# Define output directory for raw extracted parquet chunks
out_dir = Path(r"C:\Users\amina\Thesis\Final Pipeline\data_raw\iv_summary_parquet")
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Parquet chunks will be saved to:\n{out_dir}")

#Extraction chunk is defined to be 2-week long paraquets
chunk_seconds = 14 * 24 * 3600  # 14 days in seconds

for t0 in range(min_unix, max_unix + 1, chunk_seconds):
    t1 = min(t0 + chunk_seconds - 1, max_unix)

    #Define location and name of the output file
    output_file = out_dir / f"IV_Summary_{t0}_{t1}.parquet"

    # Skip existing chunks to avoid unnecessary re-extraction
    if output_file.exists():
        print(f"Skipping existing file: {output_file.name}")
        continue

    query = f"""
        SELECT *
        FROM dbo.IV_Summary
        WHERE UnixTime BETWEEN {t0} AND {t1}
        ORDER BY UnixTime, Module_ID;
    """
    #download the chunks into a pandas dataframe and save it to output directory
    df_chunk = pd.read_sql(query, conn)
    df_chunk.to_parquet(output_file, index=False)
    #print progress for extraction
    print(
        f"Saved {len(df_chunk):,} rows | "
        f"{pd.to_datetime(t0, unit='s', utc=True).date()} → "
        f"{pd.to_datetime(t1, unit='s', utc=True).date()} | "
        f"{output_file.name}"
    )

# Verify extracted files
parquet_files = sorted(out_dir.glob("IV_Summary_[0-9]*_[0-9]*.parquet"))

print(f"Total parquet chunks extracted: {len(parquet_files)}")
display([file.name for file in parquet_files[-5:]])

#close down the connection once all data has been downloaded
conn.close()
print("Database connection closed.")

## 1. Configuration

Load the project configuration and define the main input and output paths used in this notebook.

In [5]:
# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

DIRS = {
    "data_raw": PROJECT_ROOT / "data_raw",
    "data_intermediate": PROJECT_ROOT / "data_intermediate",
    "data_processed": PROJECT_ROOT / "data_processed",
    "figures": PROJECT_ROOT / "figures",
    "tables": PROJECT_ROOT / "tables",
    "logs": PROJECT_ROOT / "logs",
    "exports": PROJECT_ROOT / "exports",
    "metadata": PROJECT_ROOT / "metadata",
}

INPUT_PARQUET_DIR = DIRS["data_raw"] / "iv_summary_parquet"
RAW_BASE_OUTPUT = DIRS["data_intermediate"] / "df_raw_base.parquet"
INGEST_LOG_OUTPUT = DIRS["metadata"] / "01_ingestion_summary.json"

LOCAL_TIMEZONE = "Europe/Copenhagen"

## 2. Concatenate parquet files into one dataframe

The raw long-term monitoring data are stored as multiple parquet files. These are located and sorted before ingestion so that the read order is stable and reproducible.

In [7]:
parquet_files = sorted(INPUT_PARQUET_DIR.glob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files.")
for fp in parquet_files[:10]:
    print(fp.name)

if len(parquet_files) == 0:
    raise FileNotFoundError(
        f"No parquet files found in {INPUT_PARQUET_DIR}. "
        "Please place the long-term parquet files in this folder before continuing."
    )

Found 254 parquet files.
IV_Summary_1463162508_1464372107.parquet
IV_Summary_1464372108_1465581707.parquet
IV_Summary_1465581708_1466791307.parquet
IV_Summary_1466791308_1468000907.parquet
IV_Summary_1468000908_1469210507.parquet
IV_Summary_1469210508_1470420107.parquet
IV_Summary_1470420108_1471629707.parquet
IV_Summary_1471629708_1472839307.parquet
IV_Summary_1472839308_1474048907.parquet
IV_Summary_1474048908_1475258507.parquet


## 3. Read and combine parquet files

Each parquet file is loaded individually and then concatenated into one raw dataframe.

A per-file summary is also recorded for traceability.

In [4]:
ingestion_records = []
df_list = []

for fp in parquet_files:
    df_part = pd.read_parquet(fp)
    
    ingestion_records.append({
        "file_name": fp.name,
        "n_rows": int(df_part.shape[0]),
        "n_cols": int(df_part.shape[1]),
        "columns": list(df_part.columns),
    })
    
    df_list.append(df_part)

df_raw = pd.concat(df_list, ignore_index=True)

print("Combined raw shape:", df_raw.shape)
print("Number of source files:", len(df_list))

Combined raw shape: (9130879, 10)
Number of source files: 254


C:\Users\amina\AppData\Local\Temp\ipykernel_44184\2623180861.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_raw = pd.concat(df_list, ignore_index=True)


## 4. Initial inspection

This section checks the column structure, dtypes, and a small sample of rows before any further processing.

In [5]:
#Number of columns in the dataframe df_raw
print("Columns:")
print(df_raw.columns.tolist())
print("\n")

#datatypes of all the columns 
print("Dtypes:")
display(df_raw.dtypes.to_frame("dtype"))

#displays head of the table for initial inspection
print("\nHead:")
display(df_raw.head())

Columns:
['Module_ID', 'UnixTime', 'G', 'MPP', 'VMP', 'VOC', 'ISC', 'IMP', 'Temp', 'Change_In_Light']


Dtypes:


,dtype
Module_ID,object
UnixTime,object
G,float64
MPP,float64
VMP,float64
VOC,float64
ISC,float64
IMP,float64
Temp,float64
Change_In_Light,float64



Head:


,Module_ID,UnixTime,G,MPP,VMP,VOC,ISC,IMP,Temp,Change_In_Light
0,1,1463162508,0.018208,-0.001976,17.819813,17.819813,-0.151218,-0.000111,28.672851,-1.0
1,1,1463162522,0.018209,-0.001805,17.817652,17.818123,-0.150821,-0.000101,28.780372,-1.0
2,1,1463162536,0.018209,-0.003472,17.815169,17.815598,-0.150236,-0.000195,28.869768,-1.0
3,1,1463162551,0.018209,-0.002973,17.814545,17.814584,-0.150024,-0.000167,28.852548,-1.0
4,1,1463162565,0.018209,-0.004098,17.812669,17.812901,-0.149332,-0.000230,28.718225,-1.0


## 5. Duplicate handling

Duplicate records are removed at the raw ingestion stage using the combination of `Module_ID` and `UnixTime`, which together identify a unique module measurement timestamp.

Both the original and deduplicated row counts are recorded.

It is expected that one record is removed. 

In [6]:
n_before = len(df_raw)

df_raw = df_raw.drop_duplicates(subset=["Module_ID", "UnixTime"]).copy()

n_after = len(df_raw)
n_removed = n_before - n_after

print("Rows before deduplication :", n_before)
print("Rows after deduplication  :", n_after)
print("Duplicates removed        :", n_removed)

Rows before deduplication : 9130879
Rows after deduplication  : 9130878
Duplicates removed        : 1


## 6. Construct datetime variables

Unix timestamps are converted from UTC into timezone-aware local datetime values in `Europe/Copenhagen`.

Additional time variables are created for downstream aggregation, filtering, and plotting.

**Note:** All downstream aggregation is performed using local Danish time (`Europe/Copenhagen`) rather than raw UTC timestamps. This is necessary because daily and hourly production patterns, including potential shading signatures, must be interpreted in local solar-operational time rather than storage time.

## Timebase alignment and POA irradiance definition

The raw irradiance field `G` comes from a pyranometer mounted at the same tilt and azimuth as the PV modules.
Therefore, it is treated as plane-of-array (POA) global irradiance.

This notebook:
1. preserves the raw timestamp and raw irradiance field,
2. creates a clear alias `POA_GLOBAL`,
3. aligns all observations to a common 15-minute time bin for cross-module analysis.

In [18]:
timezone = "Europe/Copenhagen"
poa_col = "G"
bin_freq = "15min"

#convert unix time to Datetime_UTC, and also add a Datetime_Local
df_raw["Datetime_UTC"] = pd.to_datetime(df_raw["UnixTime"], unit="s", utc=True)
df_raw["Datetime_Local"] = df_raw["Datetime_UTC"].dt.tz_convert(timezone)

#irradiance is POA global in column G
df_raw["POA_GLOBAL"] = df_raw[poa_col]

## Time feature engineering
dt = df_raw["Datetime_Local"]

df_raw["Date"] = dt.dt.date
df_raw["Year"] = dt.dt.year
df_raw["Month"] = dt.dt.month
df_raw["Day"] = dt.dt.day
df_raw["Hour"] = dt.dt.hour
df_raw["Minute"] = dt.dt.minute
df_raw["DayOfYear"] = dt.dt.dayofyear
df_raw["YearMonth"] = dt.dt.to_period("M").astype(str)

C:\Users\amina\AppData\Local\Temp\ipykernel_44184\3184190450.py:22: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_raw["YearMonth"] = dt.dt.to_period("M").astype(str)


In [19]:
df_raw["Datetime_Bin_Local"] = df_raw["Datetime_Local"].dt.floor(bin_freq)
df_raw["Datetime_Bin_UTC"] = df_raw["Datetime_Bin_Local"].dt.tz_convert("UTC")

print("Binning complete:", bin_freq)

Binning complete: 15min


## Diagnostic: module alignment after binning

Check how many modules fall into each time bin.

In [20]:
bin_check = (
    df_raw.groupby("Datetime_Bin_Local")["Module_ID"]
    .nunique()
    .rename("n_modules")
    .reset_index()
)

print(bin_check["n_modules"].describe())
bin_check.head()

count    119554.000000
mean          9.221532
std           1.962407
min           1.000000
25%          10.000000
50%          10.000000
75%          10.000000
max          10.000000
Name: n_modules, dtype: float64


,Datetime_Bin_Local,n_modules
0,2016-05-13 20:00:00+02:00,1
1,2016-05-19 13:30:00+02:00,1
2,2016-05-31 13:45:00+02:00,1
3,2016-05-31 14:00:00+02:00,1
4,2016-05-31 14:15:00+02:00,1


## Diagnostic: dataset overview

In [21]:
print("Total rows:", len(df_raw))
print("Unique modules:", df_raw["Module_ID"].nunique())
print("Unique bins:", df_raw["Datetime_Bin_Local"].nunique())

Total rows: 9130878
Unique modules: 10
Unique bins: 119554


## 7. Sort and inspect time coverage

The combined dataframe is sorted by module and local time. The resulting temporal coverage is then checked at dataset level and by module.

In [22]:
df_raw = df_raw.sort_values(["Module_ID", "Datetime_Local"]).reset_index(drop=True)

dataset_time_summary = {
    "min_datetime_local": str(df_raw["Datetime_Local"].min()),
    "max_datetime_local": str(df_raw["Datetime_Local"].max()),
    "n_rows": int(df_raw.shape[0]),
    "n_modules": int(df_raw["Module_ID"].nunique()),
}

dataset_time_summary

{'min_datetime_local': '2016-05-13 20:01:48+02:00',
 'max_datetime_local': '2026-02-17 18:24:43+01:00',
 'n_rows': 9130878,
 'n_modules': 10}

In [23]:
module_time_coverage = (
    df_raw.groupby("Module_ID")
    .agg(
        min_time=("Datetime_Local", "min"),
        max_time=("Datetime_Local", "max"),
        n_rows=("Module_ID", "size"),
    )
    .reset_index()
)

display(module_time_coverage)

,Module_ID,min_time,max_time,n_rows
0,1,2016-05-13 20:01:48+02:00,2026-02-17 18:22:31+01:00,1023159
1,2,2016-09-09 15:59:40+02:00,2026-02-17 18:22:40+01:00,979378
2,3,2016-09-09 16:41:25+02:00,2026-02-17 17:45:46+01:00,753780
3,4,2016-08-24 13:36:27+02:00,2026-02-17 17:25:57+01:00,877947
4,5,2016-09-09 16:42:13+02:00,2026-02-17 18:23:24+01:00,972844
5,6,2016-09-09 16:42:37+02:00,2026-02-17 17:44:29+01:00,905674
6,7,2016-08-24 12:38:49+02:00,2026-02-17 17:36:34+01:00,872672
7,8,2016-08-24 12:38:57+02:00,2026-02-17 17:36:47+01:00,883136
8,9,2016-09-09 16:43:49+02:00,2026-02-17 17:45:13+01:00,894135
9,10,2016-09-09 16:44:13+02:00,2026-02-17 18:24:43+01:00,968153


## 8. Missingness snapshot

A basic missingness summary is produced for all columns. This is not yet used for filtering, but it helps identify variables that may need attention later in the pipeline.

In [24]:
missing_summary = (
    df_raw.isna()
    .sum()
    .rename("n_missing")
    .to_frame()
)

missing_summary["pct_missing"] = 100 * missing_summary["n_missing"] / len(df_raw)
missing_summary = missing_summary.sort_values("n_missing", ascending=False)

display(missing_summary)

,n_missing,pct_missing
Module_ID,0,0.0
POA_GLOBAL,0,0.0
Datetime_Bin_Local,0,0.0
YearMonth,0,0.0
DayOfYear,0,0.0
Minute,0,0.0
Hour,0,0.0
Day,0,0.0
Month,0,0.0
Year,0,0.0


In [25]:
#ensure that raw data extracted is present after time-feature engineering
key_columns = ["Module_ID", "UnixTime", "Datetime_Local", "G", "MPP", "VMP", "IMP", "VOC", "ISC", "Temp"]
existing_key_columns = [col for col in key_columns if col in df_raw.columns]

display(df_raw[existing_key_columns].head())

,Module_ID,UnixTime,Datetime_Local,G,MPP,VMP,IMP,VOC,ISC,Temp
0,1,1463162508,2016-05-13 20:01:48+02:00,0.018208,-0.001976,17.819813,-0.000111,17.819813,-0.151218,28.672851
1,1,1463162522,2016-05-13 20:02:02+02:00,0.018209,-0.001805,17.817652,-0.000101,17.818123,-0.150821,28.780372
2,1,1463162536,2016-05-13 20:02:16+02:00,0.018209,-0.003472,17.815169,-0.000195,17.815598,-0.150236,28.869768
3,1,1463162551,2016-05-13 20:02:31+02:00,0.018209,-0.002973,17.814545,-0.000167,17.814584,-0.150024,28.852548
4,1,1463162565,2016-05-13 20:02:45+02:00,0.018209,-0.004098,17.812669,-0.000230,17.812901,-0.149332,28.718225


## 9. Save raw base dataset

The cleaned raw base dataset is saved for downstream notebooks. This dataset includes timestamp harmonisation and duplicate removal, but no physical filtering.

Saved as `df_raw_base.parquet` in `data_intermediate` folder 
Number of columns: 23 
Number of rows: 9,130,878

In [26]:
RAW_BASE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_parquet(RAW_BASE_OUTPUT, index=False)

print(f"Saved raw base dataset to: {RAW_BASE_OUTPUT}")
print(f"Final shape: {df_raw.shape}")

Saved raw base dataset to: C:\Users\amina\Thesis\Final Pipeline\data_intermediate\df_raw_base.parquet
Final shape: (9130878, 23)


## 10. Save ingestion summary

A structured ingestion summary is saved to disk so that the raw data construction step remains traceable.

In [27]:
ingestion_summary = {
    "input_directory": str(INPUT_PARQUET_DIR),
    "n_parquet_files": len(parquet_files),
    "rows_before_deduplication": int(n_before),
    "rows_after_deduplication": int(n_after),
    "duplicates_removed": int(n_removed),
    "dataset_time_summary": dataset_time_summary,
    "source_files": ingestion_records,
    "columns_final": list(df_raw.columns),
}

with open(INGEST_LOG_OUTPUT, "w") as f:
    json.dump(ingestion_summary, f, indent=4, default=str)

print(f"Saved ingestion summary to: {INGEST_LOG_OUTPUT}")

Saved ingestion summary to: C:\Users\amina\Thesis\Final Pipeline\metadata\01_ingestion_summary.json


## 11. Final verification

A final verification is printed to confirm that the raw base dataset has been created successfully and is ready for the primary window and physical filtering notebook.

In [28]:
print("Raw base dataset created successfully.")
print("Shape:", df_raw.shape)
print("Datetime range:", df_raw["Datetime_Local"].min(), "to", df_raw["Datetime_Local"].max())
print("Unique modules:", sorted(df_raw["Module_ID"].unique().tolist()))

Raw base dataset created successfully.
Shape: (9130878, 23)
Datetime range: 2016-05-13 20:01:48+02:00 to 2026-02-17 18:24:43+01:00
Unique modules: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
